# **LangChain 기초 튜토리얼**

## **1. 실습환경 구성**

**1-1. Langchain 설치**



In [ ]:
!pip install langchain langchain-openai

from langchain_openai import OpenAIEmbeddings

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.6/99.6 kB 8.5 MB/s eta 0:00:00


**1-2. API키 설정**



In [ ]:
import os
os.environ['OPENAI_API_KEY'] = "YOUR API KEY"

## **2. LLM 체인**

**2-1. 기본 LLM 체인**

In [ ]:
from langchain_openai import ChatOpenAI

# model
llm = ChatOpenAI(model="gpt-4o-mini")

# chain 실행
print(llm.invoke("지구의 자전 주기는?").content)

지구의 자전 주기는 약 24시간입니다. 하지만 정확하게 말하자면, 태양을 기준으로 할 때의 자전 주기는 약 24시간이며, 별을 기준으로 할 경우 이를 '항성일'이라고 하여 약 23시간 56분 4초입니다. 이 두 개의 주기 간에 차이가 있는 이유는 지구가 태양 주위를 공전하기 때문입니다.


**2-2. 멀티 체인**

**기본 순차 체인**

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = init_chat_model("gpt-4o-mini")

# 2단계 순차 처리: 번역 → 자세한 한국어 설명
# 1단계: 영어를 한국어로 번역
translate_prompt = ChatPromptTemplate.from_template("다음 영어를 영어로 번역하세요: {english_word}")

# 2단계: 번역된 한국 단어 설명
explain_prompt = ChatPromptTemplate.from_template("다음 한국어 단어를 자세히 설명하세요: {korean_word}")

# 체인 1: 번역
chain1 = translate_prompt | llm | StrOutputParser()

# 체인 2: 번역 결과를 입력으로 사용
chain2 = ({"korean_word": chain1} | explain_prompt | llm | StrOutputParser())

# 실행
result = chain2.invoke({"english_word": "Artificial Intelligence"})
print(result)

"Artificial Intelligence(인공지능)"은 기계나 컴퓨터 시스템이 인간의 지능을 모방하여 학습, 사고, 문제 해결, 이해 등의 인지적 과정을 수행할 수 있도록 하는 기술이나 이론을 의미합니다. 이 용어는 일반적으로 컴퓨터 과학의 한 분야로서, 기계를 사람들이 수행하는 작업과 유사하게 수행할 수 있도록 만드는 알고리즘과 모델을 개발하는 데 초점을 맞춥니다.

### 주요 개념:

1. **기계 학습(ML, Machine Learning)**: 인공지능의 한 분야로, 데이터를 분석하여 패턴을 학습하고 예측할 수 있도록 하는 기술입니다. 기계 학습의 알고리즘은 경험을 통해 개선될 수 있습니다.

2. **딥러닝(Deep Learning)**: 기계 학습의 하위 분야로, 인공 신경망을 기반으로 한 고급 학습 방법입니다. 주로 이미지 인식, 자연어 처리 등 복잡한 데이터 분석에 사용됩니다.

3. **자연어 처리(NLP, Natural Language Processing)**: 인간의 언어를 이해하고 해석하는 AI 기술입니다. 챗봇, 번역기, 음성 인식 시스템 등이 이에 해당합니다.

4. **전문 시스템(Expert Systems)**: 특정 도메인에서 사람 전문가의 지식을 모방하여 의사 결정을 지원하는 AI 시스템입니다. 의학, 금융 등 다양한 분야에서 활용됩니다.

5. **로보틱스(Robotics)**: 인공지능을 활용하여 로봇이 환경과 상호 작용하고 자율적으로 작업을 수행할 수 있도록 하는 기술입니다.

### 적용 분야:

- **헬스케어**: 질병 진단, 맞춤형 치료 계획 수립 등을 통해 의료 서비스를 개선합니다.
- **자동차**: 자율 주행 자동차와 같은 기술에 사용되어 사고를 줄이고 효율성을 높입니다.
- **금융**: 알고리즘 거래, 위험 관리, 사기 탐지 등 다양한 금융 서비스에 활용됩니다.
- **고객 서비스**: 챗봇과 AI 도우미를 통해 고객 문의를 자동으로 처리하고 지원합니다.

### 윤리적 문제와 도전:

인공지능의 발전

**다단계 순차 체인**

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

llm = init_chat_model("gpt-4o-mini")

# 3단계 처리: 주제 분석 → 개요 작성 → 본문 작성
analyze_prompt = ChatPromptTemplate.from_template("다음 주제의 핵심 키워드 3개를 추출하세요: {topic}")

outline_prompt = ChatPromptTemplate.from_template("""다음 키워드를 바탕으로 글의 개요를 작성하세요: 키워드: {keywords} 원본 주제: {topic}""")

content_prompt = ChatPromptTemplate.from_template("""다음 개요를 바탕으로 300자 내외의 글을 작성하세요: 개요: {outline}""")

# 체인 구성
chain = ({"topic": RunnablePassthrough()}
    | RunnablePassthrough.assign(keywords=analyze_prompt | llm | StrOutputParser())
    | RunnablePassthrough.assign(outline=outline_prompt | llm | StrOutputParser())
    | content_prompt | llm | StrOutputParser())

result = chain.invoke("인공지능의 발전과 미래")
print(result)

인공지능(AI)의 발전은 오늘날 기술 혁신의 중요한 축을 이루고 있으며, 우리 삶의 여러 측면에 지대한 영향을 미치고 있습니다. AI는 초기 기계 학습에서 심층 학습으로 발전하며, 자연어 처리와 컴퓨터 비전 같은 최신 기술 동향을 통해 산업 전반과 일상생활에서 광범위하게 활용됩니다. 이러한 기술 혁신은 효율성과 편의성을 증대시키지만, 자율주행 차량과 로봇의 발전은 인간의 역할 변화와 일자리 문제를 야기하기도 합니다. 

따라서, AI 개발에서 윤리적 고려가 필수적입니다. 개인정보 보호와 알고리즘 편향 문제를 해결하기 위해 공정성과 투명성을 기준으로 한 윤리적 AI의 필요성이 대두되고 있습니다. 앞으로 AI의 발전은 기술 혁신, 자율성, 윤리적 고려의 균형을 통해 지속 가능한 미래를 이끌어 가야 할 것입니다. 이러한 관점에서 향후 연구 방향과 정책 제안이 중요할 것입니다.


**병렬 체인**

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel

llm = init_chat_model("gpt-4o-mini")

# 세 가지 관점에서 동시 분석
positive_prompt = ChatPromptTemplate.from_template("{topic}의 긍정적인 측면 3가지를 설명하세요.")

negative_prompt = ChatPromptTemplate.from_template("{topic}의 부정적인 측면 3가지를 설명하세요.")

neutral_prompt = ChatPromptTemplate.from_template("{topic}에 대한 객관적인 현황을 설명하세요.")

# 병렬 체인 구성
parallel_chain = RunnableParallel(
    positive=positive_prompt | llm | StrOutputParser(),
    negative=negative_prompt | llm | StrOutputParser(),
    neutral=neutral_prompt | llm | StrOutputParser(),
)

# 실행 (세 체인이 동시에 실행됨)
results = parallel_chain.invoke({"topic": "인공지능의 발전이 사회에 미치는 영향"})

print("=== 긍정적 측면 ===")
print(results["positive"])
print("\n=== 부정적 측면 ===")
print(results["negative"])
print("\n=== 객관적 현황 ===")
print(results["neutral"])

=== 긍정적 측면 ===
인공지능(AI)의 발전은 사회에 여러 가지 긍정적인 영향을 미치고 있습니다. 그 중에서 세 가지를 소개하겠습니다.

1. **효율성 및 생산성 향상**: AI는 다양한 산업과 분야에서 작업을 자동화하고 최적화하는 데 기여하고 있습니다. 예를 들어, 제조업에서는 로봇과 AI가 조합되어 생산라인의 효율성을 높이고, 오류를 줄이며, 생산 속도를 증가시킵니다. 또한 서비스 산업에서도 고객 서비스 챗봇과 같은 기술이 도입되어, 고객 응대의 효율성을 개선하고 인력의 부담을 줄이는 데 도움을 줍니다.

2. **데이터 분석 및 예측 능력**: AI는 방대한 양의 데이터를 처리하고 분석하는 데 뛰어난 능력을 가지고 있습니다. 이를 통해 기업과 기관은 트렌드와 패턴을 발견하고, 보다 나은 의사결정을 할 수 있습니다. 예를 들어, 의료 분야에서는 환자의 기록과 데이터를 분석하여 질병의 조기 진단 및 예측이 가능해지고, 맞춤형 치료를 제공하는 데 기여하고 있습니다.

3. **새로운 혁신과 기술 발전**: AI의 발전은 새로운 혁신과 기술을 창출하는 촉매 역할을 합니다. 인공지능을 기반으로 하는 다양한 애플리케이션과 서비스가 등장하면서, 여전히 해결되지 않은 문제들을 해결하고 새로운 시장과 직업을 창출합니다. 예를 들어, 자율주행차나 스마트 홈 기술 등은 우리의 생활 방식을 변화시키고, 새로운 산업을 형성하고 있습니다.

이러한 긍정적인 측면들은 인공지능이 사회에 미치는 영향 중 일부에 불과하며, 앞으로도 더 많은 발전과 변화가 기대됩니다.

=== 부정적 측면 ===
인공지능의 발전은 많은 긍정적인 측면이 있지만, 동시에 사회에 부정적인 영향을 미칠 수 있는 여러 가지 요소가 있습니다. 그 중에서 세 가지를 설명하겠습니다.

1. **일자리 손실과 경제적 불평등**:
   인공지능과 자동화 기술의 발전은 특정 직종에서 인간의 노동력을 대체하게 됩니다. 이로 인해 많은 일자리가 사라질 위험이 있으며, 특히 반복적이고 단순한 작업을 수행하는 직군의 노동자

##**3. 프롬프트(Prompt)**

**명확성과 구체성**

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate

llm = init_chat_model("gpt-4o-mini")

#모호한 프롬프트
vague_prompt = ChatPromptTemplate.from_template("{topic}에 대해 알려줘.")

#구체적인 프롬프트
specific_prompt = ChatPromptTemplate.from_template(
"""{topic}에 대해 다음 형식으로 설명해주세요:

1. 정의 (1-2문장)
2. 핵심 특징 3가지
3. 실제 활용 사례 2가지

전문 용어는 피하고 초보자도 이해할 수 있게 작성해주세요.""")

In [ ]:
#모호한 프롬프트에 대한 답변
chain = vague_prompt | llm
result = chain.invoke({"topic": "인공지능"})

print(result.content)

인공지능(AI, Artificial Intelligence)은 컴퓨터가 인간처럼 생각하고 학습하며 문제를 해결할 수 있도록 하는 기술 및 학문 분야입니다. 인공지능은 여러 가지 하위 분야로 나뉘며, 주요 내용은 다음과 같습니다.

1. **기계 학습(Machine Learning)**: 데이터에서 패턴을 학습하여 예측이나 결정을 내리는 알고리즘을 개발하는 분야입니다. 기계 학습은 통계학을 기반으로 하며, 지도 학습, 비지도 학습, 강화 학습 등의 방법이 있습니다.

2. **딥 러닝(Deep Learning)**: 기계 학습의 한 분야로, 인공 신경망(ANN)을 사용하여 데이터에서 높은 수준의 추상화를 학습하는 방법입니다. 주로 음성 인식, 이미지 인식, 자연어 처리 등에서 널리 사용됩니다.

3. **자연어 처리(Natural Language Processing, NLP)**: 인간의 언어를 이해하고 생성하는 방법을 연구하는 분야입니다. 챗봇, 번역기, 음성 비서 등 다양한 응용 프로그램에서 활용됩니다.

4. **컴퓨터 비전(Computer Vision)**: 컴퓨터가 이미지와 비디오를 이해하고 해석하는 기술입니다. 얼굴 인식, 객체 탐지, 이미지 분류 등 다양한 분야에 활용됩니다.

5. **로보틱스(Robotics)**: 인공지능을 이용하여 로봇이 자율적으로 동작하도록 하는 분야입니다. 물류, 제조, 의료 등 다양한 산업에서 사용됩니다.

인공지능은 의료, 금융, 교통, 교육 등 여러 산업에서 혁신적인 변화를 가져오고 있으며, 앞으로의 발전이 기대되는 분야입니다. 윤리적 문제, 개인정보 보호, 공정성 등의 이슈도 함께 다루어져야 합니다.


In [ ]:
#구체적인 프롬프트에 대한 답변
chain = specific_prompt | llm
result = chain.invoke({"topic": "인공지능"})

print(result.content)

### 1. 정의
인공지능은 컴퓨터나 기계가 인간처럼 생각하고, 배우며, 문제를 해결할 수 있도록 하는 기술입니다. 즉, 사람의 지능을 흉내 내는 프로그램이나 시스템입니다.

### 2. 핵심 특징
1. **학습 능력**: 인공지능은 많은 데이터를 통해 스스로 개선하며, 경험을 바탕으로 더 나은 결과를 만들어냅니다.
2. **패턴 인식**: 인공지능은 다양한 정보에서 규칙이나 패턴을 찾아내어 이를 바탕으로 예측이나 결정을 할 수 있습니다.
3. **적응성**: 변화하는 환경이나 상황에 맞춰 스스로 조정하며 새로운 정보에 빠르게 반응할 수 있습니다.

### 3. 실제 활용 사례
1. **가상 비서**: 스마트폰의 음성 비서인 Siri나 Google Assistant는 사용자의 음성을 이해하고, 일정 관리, 정보 검색 등을 도와줍니다.
2. **스마트 추천 시스템**: 넷플릭스나 유튜브는 사용자가 좋아할 만한 콘텐츠를 추천하기 위해 인공지능을 사용하여 시청 기록과 선호도를 분석합니다.


**컨텍스트(context) 제공**

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# 컨텍스트 없는 프롬프트
without_context = "파이썬 웹 프레임워크를 추천해줘."

# 컨텍스트 포함 프롬프트
with_context = ChatPromptTemplate.from_messages([
    ("system", "당신은 시니어 백엔드 개발자입니다."),
    ("human", """다음 프로젝트 요구사항에 맞는 파이썬 웹 프레임워크를 추천해주세요.

프로젝트 정보:
- 팀 규모: 3명 (주니어 2, 시니어 1)
- 예상 트래픽: 일 10만 요청
- 주요 기능: REST API, 실시간 알림
- 기한: 3개월

각 프레임워크의 장단점과 이 프로젝트에 적합한 이유를 설명해주세요.""")
])

In [ ]:
#컨텍스트가 없는 프롬프트 대한 답변
result = chain.invoke(without_context)
print(result.content)

파이썬에는 여러 가지 웹 프레임워크가 있으며, 각각의 프레임워크는 특정한 요구 사항과 프로젝트에 적합합니다. 아래는 인기 있는 파이썬 웹 프레임워크 몇 가지를 소개합니다.

1. **Django**:
   - 고급 기능이 많이 포함된 웹 프레임워크로, "배터리 포함" 방식을 채택하고 있습니다. 
   - 사용자 인증, 관리 패널, ORM, URL 라우팅 등 많은 기능을 기본 제공.
   - 대규모 프로젝트에 적합하며, 보안 측면에서도 강력합니다.

2. **Flask**:
   - 경량화된 웹 프레임워크로, 매우 유연하고 간단합니다.
   - 기본 기능만 포함되어 있어 필요한 기능을 플러그인 형태로 추가할 수 있습니다.
   - 작은 애플리케이션에서부터 중간 규모의 애플리케이션까지 폭넓게 사용됩니다.

3. **FastAPI**:
   - 현대적인 비동기 웹 프레임워크로, 빠른 성능과 자동화된 API 문서화를 제공합니다.
   - ASGI를 기반으로 하며 비동기 프로그래밍을 지원합니다.
   - RESTful API 및 웹소켓 서버 구축에 유리합니다.

4. **Tornado**:
   - 비동기 네트워크 라이브러리와 웹 프레임워크가 결합된 형태로, 높은 성능을 자랑합니다.
   - 실시간 웹 애플리케이션을 구축하기에 적합합니다(예: 채팅 앱).

5. **Pyramid**:
   - 유연성과 확장성을 중시하는 웹 프레임워크로, 작은 프로젝트에서부터 큰 애플리케이션까지 적합합니다.
   - 필요에 따라 다양한 구성 요소를 선택하여 사용할 수 있습니다.

6. **Bottle**:
   - 매우 간단하고 경량화된 마이크로 프레임워크입니다.
   - 단일 파일로 구성되어 있어 작은 프로젝트나 프로토타입을 빠르게 구축할 때 유용합니다.

프로젝트의 규모, 요구 사항, 팀의 경험 등을 고려하여 적합한 프레임워크를 선택하는 것이 중요합니다. 각 프레임워크의 문서와 튜토리얼을 참고하여 필요에 맞는 것을 선택하세요.


In [ ]:
#컨텍스트가 있는 프롬프트 대한 답변
result = chain.invoke(with_context)
print(result.content)

물론입니다! 주어진 정보를 바탕으로, 해당 프로젝트 요구사항에 적합한 파이썬 웹 프레임워크를 추천하겠습니다. 프로젝트의 요구사항은 다음과 같습니다:

1. 팀 규모: 3명 (주니어 2, 시니어 1)
2. 예상 트래픽: 일 10만 요청
3. 주요 기능: REST API, 실시간 알림
4. 기한: 3개월

여기에 적합한 파이썬 웹 프레임워크 몇 가지를 추천하겠습니다.

### 1. Django
- **장점:**
  - **완벽한 기능 세트:** Django는 성숙한 프레임워크로, ORM(Object-Relational Mapping), 인증, 관리자 패널, 폼 처리 등 많은 기능을 기본 제공함.
  - **대규모 트래픽 처리:** Django는 높은 트래픽을 처리할 수 있도록 설계되었음.
  - **설정이 용이함:** REST API를 만들기 위해 Django REST Framework를 사용할 수 있어 RESTful 서비스 구축이 수월함.
  
- **단점:**
  - **무겁고 복잡할 수 있음:** 큰 프로젝트를 위한 다양한 기능을 포함하고 있지만, 간단한 애플리케이션에는 과도할 수 있음.
  - **학습 곡선:** 특히 주니어 개발자에게는 학습 곡선이 가파를 수 있음.

- **적합성:** 이 프로젝트에서 Django는 팀의 크기와 예상 트래픽을 충분히 처리할 수 있으며, REST API와 실시간 알림 기능을 손쉽게 구현할 수 있습니다.

### 2. Flask
- **장점:**
  - **경량성:** Flask는 본질적으로 간단한 웹 프레임워크로 필요한 기능을 최소한으로 제공함.
  - **유연성:** 사용자가 필요한 라이브러리나 기능을 쉽게 추가할 수 있어 필요에 맞는 구조를 만들 수 있음.
  - **학습 용이성:** Flask는 직관적이며 배우기 쉬워 주니어 개발자에게 적합.

- **단점:**
  - **기본 제공 기능 부족:** Django에 비해 기능이 적어 직접 몇 가지 기능을 구현해야 할 수 있음.
  - **스케일링 문제:** 대규모 애플리

**구조화된 프롬프트**

In [ ]:
#역할(ROLE 설정)
from langchain_core.prompts import ChatPromptTemplate

llm = init_chat_model("gpt-4o-mini")

# System 메시지로 역할 정의
prompt = ChatPromptTemplate.from_messages([
    ("system", """당신은 10년 경력의 데이터 사이언티스트입니다.

전문 분야:
- 머신러닝 모델 개발
- 데이터 분석 및 시각화
- Python, SQL, TensorFlow

응답 스타일:
- 데이터 기반 설명
- 코드 예시 포함
- 실무 관점의 조언"""),
    ("human", "{question}")
])
question = "python을 기반으로 데이터를 시각화 하기 위한 가장 적절한 방법을 알려줘"

chain = prompt | llm

result = chain.invoke({"question": question})

print(result.content)

데이터 시각화는 데이터 분석에서 중요한 단계로, 데이터를 시각적으로 표현하여 인사이트를 도출하는 데 도움을 줍니다. Python에서는 여러 시각화 라이브러리를 사용할 수 있지만, 가장 적절하고 일반적으로 사용되는 방법은 다음과 같습니다:

1. **Matplotlib**: 기본적인 시각화 도구로, 다양한 유형의 그래프를 그릴 수 있습니다.
2. **Seaborn**: Matplotlib을 바탕으로 더 아름답고 복잡한 통계적 그래프를 쉽게 만들 수 있는 라이브러리입니다.
3. **Plotly**: 인터랙티브한 그래프를 만들 수 있는 라이브러리로, 웹 애플리케이션이나 대시보드에서 자주 사용됩니다.

### 기본 예시

여기서는 `Matplotlib`와 `Seaborn`을 사용하여 데이터를 시각화하는 방법을 보여드리겠습니다.

#### 1. Matplotlib을 이용한 시각화

```python
import matplotlib.pyplot as plt
import numpy as np

# 예제 데이터 생성
x = np.linspace(0, 10, 100)
y = np.sin(x)

# 그래프 그리기
plt.figure(figsize=(10, 5))
plt.plot(x, y, label='Sine Wave', color='blue')
plt.title('Sine Wave')
plt.xlabel('X-axis')
plt.ylabel('Y-axis')
plt.legend()
plt.grid()
plt.show()
```

#### 2. Seaborn을 이용한 시각화

```python
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

# 예제 데이터프레임 생성
data = pd.DataFrame({
    'Category': ['A', 'B', 'C', 'D'],
    'Values': [25, 40, 35, 10]
})

# 바 차트 그리기
plt.figure(figsize=(8,

**출력 형식 지정**

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 표 형식 요청
table_prompt = ChatPromptTemplate.from_messages([
    ("system", "응답은 마크다운 표 형식으로 작성하세요."),
    ("human", """다음 프로그래밍 언어들을 비교해주세요: {languages}

| 언어 | 주요 용도 | 장점 | 단점 | 학습 난이도 |
형식으로 작성해주세요.""")
])

# JSON 형식 요청
json_prompt = ChatPromptTemplate.from_messages([
    ("system", "응답은 유효한 JSON 형식으로만 작성하세요. 다른 텍스트는 포함하지 마세요."),
    ("human", """다음 텍스트에서 정보를 추출하세요: {text}

형식:
{{
    "name": "이름",
    "email": "이메일",
    "phone": "전화번호"
}}""")
])

# 번호 목록 형식 요청
list_prompt = ChatPromptTemplate.from_messages([
    ("system", "응답은 번호 목록으로 작성하세요. 각 항목은 한 줄로 간결하게."),
    ("human", "{topic}의 장점 5가지를 알려주세요.")
])

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = init_chat_model("gpt-4o-mini")

# 표 형식 요청
table_prompt = ChatPromptTemplate.from_messages([
    ("system", "응답은 마크다운 표 형식으로 작성하세요."),
    ("human", """다음 프로그래밍 언어들을 비교해주세요: {languages}

| 언어 | 주요 용도 | 장점 | 단점 | 학습 난이도 |
형식으로 작성해주세요.""")
])

languages = "python, java, c++"

chain = table_prompt | llm

result = chain.invoke({"languages": languages})

print(result.content)

| 언어   | 주요 용도                | 장점                                                       | 단점                                                      | 학습 난이도 |
|--------|-------------------------|----------------------------------------------------------|----------------------------------------------------------|------------|
| Python | 데이터 과학, 웹 개발, 인공지능, 자동화 | - 문법이 간단하고 가독성이 높음<br>- 다양한 라이브러리와 프레임워크 지원<br>- 커뮤니티가 활발함 | - 실행 속도가 느림<br>- 모바일 앱 개발에 적합하지 않음 | 낮음       |
| Java   | 기업용 애플리케이션, 웹 서버, 안드로이드 앱 | - 강력한 객체 지향 언어<br>- 크로스 플랫폼 전송 가능 (JVM)<br>- 안정성이 높음 | - 복잡한 문법<br>- 메모리 관리가 필요할 수 있음      | 중간     |
| C++    | 게임 개발, 시스템 프로그래밍, 고성능 애플리케이션 | - 성능이 매우 좋음<br>- 시스템 자원에 대한 직접적인 접근 가능<br>- 유연한 메모리 관리 | - 복잡한 문법<br>- 메모리 관리로 인한 오류 가능성    | 높음      |


**예시 제공 (Few-shot)**

In [ ]:
#프롬프트 템플릿 실습
from langchain_core.prompts import PromptTemplate

# 'name'과 'age'라는 두 개의 변수를 사용하는 프롬프트 템플릿을 정의
template_text = "안녕하세요, 제 이름은 {name}이고, 나이는 {age}살입니다."

# PromptTemplate 인스턴스를 생성
prompt_template = PromptTemplate.from_template(template_text)

# 템플릿에 값을 채워서 프롬프트를 완성
filled_prompt = prompt_template.format(name="홍길동", age=30)

print(filled_prompt)

안녕하세요, 제 이름은 홍길동이고, 나이는 30살입니다.


In [ ]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-4o-mini", temperature=0.0)

# 모델에 질문하기
result = model.invoke("지구의 자전 주기는 얼마인가요?")
print(result.content)

지구의 자전 주기는 약 24시간입니다. 정확히 말하면, 지구가 한 번 자전하는 데 걸리는 시간은 약 23시간 56분 4초로, 이를 '항성일'이라고 합니다. 그러나 우리가 일반적으로 사용하는 24시간은 태양이 하늘에서 같은 위치에 돌아오는 데 걸리는 시간을 기준으로 한 '태양일'입니다.


In [ ]:
# 고정된 예제 사용하기
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate

# 예제 정의
examples = [
    {"input": "지구의 대기 중 가장 많은 비율을 차지하는 기체는 무엇인가요?", "output": "질소입니다."},
    {"input": "광합성에 필요한 주요 요소들은 무엇인가요?", "output": "빛, 이산화탄소, 물입니다."},
]

# 예제 프롬프트 템플릿 정의
example_prompt = ChatPromptTemplate.from_messages([("human", "{input}"), ("ai", "{output}"),])

# Few-shot 프롬프트 템플릿 생성
few_shot_prompt = FewShotChatMessagePromptTemplate(example_prompt=example_prompt, examples=examples,)

# 최종 프롬프트 템플릿 생성
final_prompt = ChatPromptTemplate.from_messages([("system", "당신은 과학과 수학에 대해 잘 아는 교육자입니다."), few_shot_prompt, ("human", "{input}"),])

# 모델과 체인 생성
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-4o-mini", temperature=0.0)
chain = final_prompt | model

# 모델에 질문하기
result = chain.invoke({"input": "태양계에서 가장 큰 행성은 무엇인가요?"})
print(result.content)

태양계에서 가장 큰 행성은 목성(Jupiter)입니다.


In [ ]:
# 동적 few-shot prompting
from langchain_chroma import Chroma
from langchain_core.example_selectors import SemanticSimilarityExampleSelector
from langchain_core.prompts import FewShotChatMessagePromptTemplate, ChatPromptTemplate
from langchain_openai import OpenAIEmbeddings, ChatOpenAI

# 더 많은 예제 추가
examples = [
    {"input": "지구의 대기 중 가장 많은 비율을 차지하는 기체는 무엇인가요?", "output": "질소입니다."},
    {"input": "광합성에 필요한 주요 요소들은 무엇인가요?", "output": "빛, 이산화탄소, 물입니다."},
    {"input": "피타고라스 정리를 설명해주세요.", "output": "직각삼각형에서 빗변의 제곱은 다른 두 변의 제곱의 합과 같습니다."},
    {"input": "DNA의 기본 구조를 간단히 설명해주세요.", "output": "DNA는 이중 나선 구조를 가진 핵산입니다."},
    {"input": "원주율(π)의 정의는 무엇인가요?", "output": "원의 둘레와 지름의 비율입니다."},
    {"input": "목성의 부피는 지구의 몇 배인가요?", "output": "약 1,321배 입니다."}
]

# 벡터 저장소 생성
to_vectorize = [" ".join(example.values()) for example in examples]
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = Chroma.from_texts(to_vectorize, embeddings, metadatas=examples)

# 예제 선택기 생성
example_selector = SemanticSimilarityExampleSelector(vectorstore=vectorstore, k=2,)

# Few-shot 프롬프트 템플릿 생성
few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_selector=example_selector,
    example_prompt=ChatPromptTemplate.from_messages(
        [("human", "{input}"), ("ai", "{output}")]
    ),
)

# 최종 프롬프트 템플릿 생성
final_prompt = ChatPromptTemplate.from_messages([("system", "당신은 과학과 수학에 대해 잘 아는 교육자입니다."), few_shot_prompt, ("human", "{input}"),])

# 모델과 체인 생성
chain = final_prompt | ChatOpenAI(model="gpt-4o-mini", temperature=0.0)

question = "태양계에서 가장 큰 행성은 무엇인가요?"

selected_examples = example_selector.select_examples({"input": question})

print("=== 선택된 예제 ===")
for i, example in enumerate(selected_examples, 1):
    print(f"[Example {i}]")
    print("Input :", example["input"])
    print("Output:", example["output"])
    print()


# 모델에 질문하기
result = chain.invoke(question)
print(result.content)

=== 선택된 예제 ===
[Example 1]
Input : 목성의 부피는 지구의 몇 배인가요?
Output: 약 1,321배 입니다.

[Example 2]
Input : 피타고라스 정리를 설명해주세요.
Output: 직각삼각형에서 빗변의 제곱은 다른 두 변의 제곱의 합과 같습니다.

태양계에서 가장 큰 행성은 목성(Jupiter)입니다. 목성은 지구보다 훨씬 크고, 그 크기와 질량으로 인해 태양계의 다른 행성들에 비해 압도적인 존재감을 가지고 있습니다.


## **4. Memory**

In [ ]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langgraph.checkpoint.memory import MemorySaver

# 에이전트 생성 - checkpointer를 직접 전달
model = init_chat_model("openai:gpt-4o")

without_memory1 = model.invoke("안녕하세요, 제 이름은 영희입니다.")
print(without_memory1.content)

without_memory2 = model.invoke("제 이름이 뭐죠?")
print(without_memory2.content)

안녕하세요, 영희님! 만나서 반갑습니다. 어떻게 도와드릴까요?
죄송하지만, 제가 사용자님의 이름을 알 수는 없습니다. 개인정보를 제공하지 않으셔도 질문에 답변드리거나 필요한 정보를 제공해드릴 수 있습니다. 어떤 도움이 필요하신가요?


In [ ]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langgraph.checkpoint.memory import MemorySaver

# Checkpointer 생성 (메모리 기반)
checkpointer = MemorySaver()

# 에이전트 생성 - checkpointer를 직접 전달
model = init_chat_model("openai:gpt-4o")
agent = create_agent(
    model,
    tools=[],
    checkpointer=checkpointer  # 메모리 활성화
)

# 대화 실행 - agent.invoke() 직접 호출
config = {"configurable": {"thread_id": "conversation-1"}}

result1 = agent.invoke(
    {"messages": [("user", "안녕하세요, 제 이름은 영희입니다.")]},
    config=config
)
print(result1["messages"][-1].content)

result2 = agent.invoke(
    {"messages": [("user", "제 이름이 뭐죠?")]},
    config=config
)
print(result2["messages"][-1].content)

안녕하세요, 영희님! 만나서 반갑습니다. 오늘 어떻게 도와드릴까요?
당신은 조금 전에 본인의 이름이 영희라고 말씀하셨습니다. 맞나요?


## **5. 챗봇 (Chatbot) 만들기**

In [ ]:
# ── 에이전트 생성 (Prompt Engineering + Checkpointer) ──
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langgraph.checkpoint.memory import MemorySaver

# 시스템 프롬프트로 역할/말투/제약 지정
SYSTEM_PROMPT = """당신은 'SciBot'이라는 과학 기반 AI 연구 도우미입니다.
당신의 목표는 연구자가 과학적 사고를 바탕으로 정확하고 신뢰할 수 있는 답을 얻도록 돕는 것입니다.

[역할]
- 물리, 화학, 생물, 데이터과학, AI/머신러닝 등 과학 전반의 연구 질문을 돕습니다.
- 가설 수립, 실험 설계, 논문 해석, 결과 분석, 방법론 선택을 지원합니다.
- 단순 정답 제공자가 아니라, 연구자의 추론 과정을 함께 점검하는 동료처럼 행동합니다.

[답변 원칙]
1. 과학적 근거 우선: 주장에는 그 근거(원리, 메커니즘, 데이터)를 함께 제시합니다.
2. 정확성 우선: 확실한 것과 불확실한 것을 명확히 구분하고, 모르거나 합의되지 않은 내용은 솔직하게 밝힙니다.
3. 추측 금지: 검증되지 않은 내용을 사실처럼 단정하지 않으며, 추정일 경우 "추정"임을 명시합니다.
4. 정량적 사고: 가능하면 수식, 단위, 크기 규모(order of magnitude)를 활용해 설명합니다.
5. 한계 명시: 방법이나 결론의 가정과 한계, 반례 가능성을 함께 언급합니다.

[추론 방식]
- 복잡한 문제는 단계적으로 분해해 논리적으로 설명합니다.
- 가정 → 추론 → 결론의 흐름을 분명히 드러냅니다.
- 여러 해석이 가능할 경우, 주요 가설들을 비교하고 각각의 근거와 약점을 제시합니다.

[출력 형식]
- 한국어로, 전문 용어는 처음 등장할 때 간단한 정의를 덧붙입니다.
- 수식은 명확한 표기로, 코드는 핵심 부분에 짧은 주석을 답니다.
- 답변은 핵심을 먼저 제시하고, 필요한 경우 근거와 세부 설명을 이어갑니다.
- 불확실하거나 검증이 필요한 부분은 마지막에 "검증 필요" 또는 "추가 확인 권장"으로 표시합니다.

[태도]
- 이전 대화 맥락을 기억하고 일관되게 이어서 답합니다.
- 사용자가 잘못된 전제를 가졌다면 정중하게 바로잡되, 그 이유를 함께 설명합니다.
- 비판적이되 건설적인 연구 동료의 태도를 유지합니다.
"""

checkpointer = MemorySaver()          # 단기 메모리
model = init_chat_model("openai:gpt-4o")

agent = create_agent(
    model,
    tools=[],
    system_prompt=SYSTEM_PROMPT,      # 프롬프트 엔지니어링 적용
    checkpointer=checkpointer,        # 메모리 활성화
)

In [ ]:
config = {"configurable": {"thread_id": "chat-session"}}

while True:
    user_input = input("나: ")
    if user_input.strip().lower() in ("exit", "quit", "종료"):
        break
    result = agent.invoke(
        {"messages": [("user", user_input)]},
        config=config,
    )
    print("LangBot:", result["messages"][-1].content)

나: 안녕 내 이름은 철수고 인공지능을 전공했어, 앞으로 내가 연구를 수행함에 있어 도움을 주면 좋겠어
LangBot: 안녕하세요, 철수님! 만나서 반갑습니다. 인공지능 분야에서 연구를 준비하고 계시다니 흥미롭습니다. 어떤 주제나 문제를 탐구하고 계신지, 혹은 도움을 필요로 하는 특정 부분이 있는지 말씀해주시면 도움이 될 수 있도록 최선을 다하겠습니다. 함께 과학적 사고를 통해 목표를 이루어가요!
나: 내가 전공한 분야가 뭐라고?
LangBot: 철수님은 인공지능을 전공하셨다고 말씀하셨어요. 인공지능 분야의 연구에 도움을 드리도록 하겠습니다. 특정한 질문이나 주제가 있으면 편하게 말씀해 주세요.
나: LLM에 대해서 설명해줄래?
LangBot: 물론입니다! LLM은 "Large Language Model"의 약자로, 대규모 언어 모델을 의미합니다. 이는 인간의 언어를 이해하고 생성하기 위해 설계된 인공지능 모델입니다. 다음은 LLM의 구성과 기능에 대한 간단한 설명입니다:

1. **기본 원리**:
   - LLM은 주로 딥러닝(deep learning) 기술을 사용하며, 자연어 처리(NLP)를 위한 트랜스포머(transformer) 아키텍처를 기반으로 합니다. 트랜스포머는 병렬화(parallelization)와 문맥을 고려한 계산을 가능하게 하여 언어 이해에 효과적입니다.

2. **훈련 데이터**:
   - LLM은 인터넷, 책, 기사 등에서 수집한 대규모 텍스트 데이터를 사용하여 학습합니다. 이를 통해 다양한 문맥과 주제에 대해 방대한 양의 정보를 습득할 수 있습니다.

3. **모델 크기**:
   - '대규모'라는 이름처럼, 매개변수(parameter)의 수가 수십억에서 수천억 개에 이릅니다. 이는 모델이 더 복잡한 패턴과 의미를 학습할 수 있도록 도와줍니다.

4. **응용 분야**:
   - 텍스트 생성, 번역, 요약, 질문-응답 시스템, 감정 분석 등 다양한 응용 가능성이 있습니다. LLM은 고급 문장 구조를 이해하고 적절한 대응을 생성하는 데 

KeyboardInterrupt: Interrupted by user

In [ ]:
# ── 저장된 메모리 확인 ──
state = agent.get_state(config)
for m in state.values["messages"]:
    m.pretty_print()

================================ Human Message =================================

안녕 내 이름은 철수고 인공지능을 전공했어, 앞으로 내가 연구를 수행함에 있어 도움을 주면 좋겠어
================================== Ai Message ==================================

안녕하세요, 철수님! 만나서 반갑습니다. 인공지능 분야에서 연구를 준비하고 계시다니 흥미롭습니다. 어떤 주제나 문제를 탐구하고 계신지, 혹은 도움을 필요로 하는 특정 부분이 있는지 말씀해주시면 도움이 될 수 있도록 최선을 다하겠습니다. 함께 과학적 사고를 통해 목표를 이루어가요!
================================ Human Message =================================

내가 전공한 분야가 뭐라고?
================================== Ai Message ==================================

철수님은 인공지능을 전공하셨다고 말씀하셨어요. 인공지능 분야의 연구에 도움을 드리도록 하겠습니다. 특정한 질문이나 주제가 있으면 편하게 말씀해 주세요.
================================ Human Message =================================

LLM에 대해서 설명해줄래?
================================== Ai Message ==================================

물론입니다! LLM은 "Large Language Model"의 약자로, 대규모 언어 모델을 의미합니다. 이는 인간의 언어를 이해하고 생성하기 위해 설계된 인공지능 모델입니다. 다음은 LLM의 구성과 기능에 대한 간단한 설명입니다:

1. **기본 원리**:
   - LLM은 주로 딥러닝(deep learning) 기술